In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

25/03/03 21:48:13 WARN Utils: Your hostname, LAPTOP-5QMII800 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/03/03 21:48:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/03 21:48:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df_green = spark.read.parquet('data/pq/green/*/*')

```
SELECT 
    date_trunc('hour', lpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
```

In [3]:
rdd = df_green \
    .select('lpep_pickup_datetime', 'PULocationID', 'total_amount') \
    .rdd

In [4]:
rdd.take(5)

[Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 14, 13, 28, 9), PULocationID=74, total_amount=9.36),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 28, 12, 11, 44), PULocationID=129, total_amount=15.3),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 4, 11, 38), PULocationID=25, total_amount=28.88),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 30, 10, 3, 36), PULocationID=182, total_amount=12.3),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 30, 20, 56, 17), PULocationID=210, total_amount=7.0)]

In [5]:
from datetime import datetime

In [6]:
start = datetime(year=2021, month=1, day=1)

def filter_outliers(row):
    return row.lpep_pickup_datetime >= start

In [7]:
rows = rdd.take(10)
row = rows[0]

In [8]:
row

Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 14, 13, 28, 9), PULocationID=74, total_amount=9.36)

In [9]:
def prepare_for_grouping(row): 
    hour = row.lpep_pickup_datetime.replace(minute=0, second=0, microsecond=0)
    zone = row.PULocationID
    key = (hour, zone)
    
    amount = row.total_amount
    count = 1
    value = (amount, count)

    return (key, value)

In [10]:
def calculate_revenue(left_value, right_value):
    left_amount, left_count = left_value
    right_amount, right_count = right_value
    
    output_amount = left_amount + right_amount
    output_count = left_count + right_count
    
    return (output_amount, output_count)

In [11]:
from collections import namedtuple

In [12]:
RevenueRow = namedtuple('RevenueRow', ['hour', 'zone', 'revenue', 'count'])

In [13]:
def unwrap(row):
    return RevenueRow(
        hour=row[0][0], 
        zone=row[0][1],
        revenue=row[1][0],
        count=row[1][1]
    )

In [14]:
from pyspark.sql import types

In [15]:
result_schema = types.StructType([
    types.StructField('hour', types.TimestampType(), True), 
    types.StructField('zone', types.IntegerType(), True), 
    types.StructField('revenue', types.DoubleType(), True), 
    types.StructField('count', types.IntegerType(), True)
])

In [16]:
df_result = rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .toDF(result_schema)

In [ ]:
df_result.write.parquet('tmp/green-revenue')

In [19]:
columns = ['VendorID', 'lpep_pickup_datetime', 'PULocationID', 'DOLocationID', 'trip_distance']

duration_rdd = df_green \
    .select(columns) \
    .rdd

In [22]:
import pandas as pd

In [23]:
rows = duration_rdd.take(10)

In [26]:
pd.DataFrame(rows, columns = columns)

,VendorID,lpep_pickup_datetime,PULocationID,DOLocationID,trip_distance
0,2.0,2020-01-14 13:28:09,74,75,1.35
1,2.0,2020-01-28 12:11:44,129,179,3.73
2,NaN,2020-01-04 11:38:00,25,41,11.19
3,2.0,2020-01-30 10:03:36,182,242,1.32
4,1.0,2020-01-30 20:56:17,210,210,0.80
5,2.0,2020-01-06 14:40:15,33,66,1.24
6,2.0,2020-01-09 18:01:49,42,116,0.52
7,2.0,2020-01-29 04:39:21,116,116,0.58
8,NaN,2020-01-09 15:01:00,223,92,6.79
9,2.0,2020-01-31 08:32:25,55,55,9.69


In [29]:
columns

['VendorID',
 'lpep_pickup_datetime',
 'PULocationID',
 'DOLocationID',
 'trip_distance']

In [30]:
#model = ...

def model_predict(df):
#     y_pred = model.predict(df)
    y_pred = df.trip_distance * 5
    return y_pred

In [36]:
def apply_model_in_batch(rows):
    df = pd.DataFrame(rows, columns=columns)
    df['lpep_pickup_datetime'] = df['lpep_pickup_datetime'].astype(str) # string conversion
    predictions = model_predict(df)
    df['predicted_duration'] = predictions

    for row in df.itertuples():
        yield row

In [37]:
df_predicts = duration_rdd \
    .mapPartitions(apply_model_in_batch)\
    .toDF() \
    .drop('Index')

In [39]:
df_predicts.show()

[Stage 14:>                                                         (0 + 1) / 1]

+--------+--------------------+------------+------------+-------------+------------------+
|VendorID|lpep_pickup_datetime|PULocationID|DOLocationID|trip_distance|predicted_duration|
+--------+--------------------+------------+------------+-------------+------------------+
|     2.0| 2020-01-14 13:28:09|          74|          75|         1.35|              6.75|
|     2.0| 2020-01-28 12:11:44|         129|         179|         3.73|             18.65|
|     NaN| 2020-01-04 11:38:00|          25|          41|        11.19|55.949999999999996|
|     2.0| 2020-01-30 10:03:36|         182|         242|         1.32|6.6000000000000005|
|     1.0| 2020-01-30 20:56:17|         210|         210|          0.8|               4.0|
|     2.0| 2020-01-06 14:40:15|          33|          66|         1.24|               6.2|
|     2.0| 2020-01-09 18:01:49|          42|         116|         0.52|               2.6|
|     2.0| 2020-01-29 04:39:21|         116|         116|         0.58|               2.9|